# Graph-Augmented Intelligence
## Notebook 04: Train & Evaluate ML Models

With graph features written to the Feature Store in **`03_pull_gold_tables`**, this notebook
trains two models on the same fraud-detection task and measures the lift that
graph features provide.

```
Feature Store ──► Training Table ──► Baseline Model
                                  └──► Graph-Augmented Model
```

## 0. Configuration

In [ ]:
CATALOG   = "graph-enriched-lakehouse"
SCHEMA    = "graph-enriched-schema"

NEO4J_URI      = dbutils.secrets.get("neo4j-graph-engineering", "uri")
NEO4J_USER     = dbutils.secrets.get("neo4j-graph-engineering", "username")
NEO4J_PASSWORD = dbutils.secrets.get("neo4j-graph-engineering", "password")

NEO4J_OPTS = {
    "url":                    NEO4J_URI,
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASSWORD,
}

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

TRAINING_TABLE = f"`{CATALOG}`.`{SCHEMA}`.training_dataset"

In [ ]:
from pyspark.sql import functions as F

In [ ]:
training_df = spark.table(TRAINING_TABLE)
print(f"Loaded {training_df.count()} rows from {TRAINING_TABLE}")

---
## 1. Prepare Data for Modelling

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

mlflow.set_experiment(f"/Workspace/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/graph_augmented_fraud")

In [ ]:
pdf = training_df.toPandas()

# Define feature sets
TABULAR_FEATURES = [
    "balance", "holder_age",
    "txn_count", "avg_txn_amount", "std_txn_amount", "max_txn_amount",
    "unique_merchants", "avg_txn_hour", "night_txn_ratio",
    "p2p_out_count", "avg_p2p_amount",
]

GRAPH_FEATURES = [
    "risk_score",
    "community_id",
    "similarity_score",
]

ALL_FEATURES = TABULAR_FEATURES + GRAPH_FEATURES
LABEL = "is_fraud"

# Clean
pdf = pdf.fillna(0)
pdf["is_fraud"] = pdf["is_fraud"].astype(int)

# Encode categoricals as ordinals for the demo
pdf["account_type_enc"] = pdf["account_type"].astype("category").cat.codes
pdf["region_enc"]       = pdf["region"].astype("category").cat.codes
TABULAR_FEATURES += ["account_type_enc", "region_enc"]
ALL_FEATURES     += ["account_type_enc", "region_enc"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    pdf[ALL_FEATURES], pdf[LABEL], test_size=0.2, random_state=42, stratify=pdf[LABEL]
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Fraud rate (train): {y_train.mean():.3f}")

---
## 2. Train Baseline Model (Tabular Only)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve,
    ConfusionMatrixDisplay,
)

def train_and_log(model_name, feature_cols, X_train, X_test, y_train, y_test):
    """Train a GBM, log to MLflow, return model and metrics."""

    with mlflow.start_run(run_name=model_name):
        clf = GradientBoostingClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.8,
            random_state=42,
        )
        clf.fit(X_train[feature_cols], y_train)
        y_pred = clf.predict(X_test[feature_cols])
        y_prob = clf.predict_proba(X_test[feature_cols])[:, 1]

        metrics = {
            "auc":       roc_auc_score(y_test, y_prob),
            "precision": precision_score(y_test, y_pred),
            "recall":    recall_score(y_test, y_pred),
            "f1":        f1_score(y_test, y_pred),
        }

        mlflow.log_param("model_type", "GradientBoosting")
        mlflow.log_param("features", model_name)
        mlflow.log_param("n_features", len(feature_cols))
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(clf, artifact_path="model")

        print(f"\n{'='*60}")
        print(f"  {model_name}")
        print(f"{'='*60}")
        for k, v in metrics.items():
            print(f"  {k:>12s}: {v:.4f}")

        return clf, metrics

In [ ]:
baseline_clf, baseline_metrics = train_and_log(
    "Baseline (Tabular Only)",
    TABULAR_FEATURES,
    X_train, X_test, y_train, y_test,
)

## 3. Train Graph-Augmented Model

In [ ]:
graph_clf, graph_metrics = train_and_log(
    "Graph-Augmented (Tabular + Neo4j GDS)",
    ALL_FEATURES,
    X_train, X_test, y_train, y_test,
)

---
## 4. Head-to-Head Comparison

In [ ]:
comparison = pd.DataFrame({
    "Metric":          ["AUC", "Precision", "Recall", "F1"],
    "Baseline":        [baseline_metrics["auc"], baseline_metrics["precision"],
                        baseline_metrics["recall"], baseline_metrics["f1"]],
    "Graph-Augmented": [graph_metrics["auc"], graph_metrics["precision"],
                        graph_metrics["recall"], graph_metrics["f1"]],
})
comparison["Lift"] = comparison["Graph-Augmented"] - comparison["Baseline"]
comparison["Lift %"] = (comparison["Lift"] / comparison["Baseline"] * 100).round(1)

display(spark.createDataFrame(comparison))

## 5. Feature Importance — What Drives the Graph-Augmented Model?

In [ ]:
importances = pd.DataFrame({
    "feature":    ALL_FEATURES,
    "importance": graph_clf.feature_importances_,
}).sort_values("importance", ascending=False)

display(spark.createDataFrame(importances.head(15)))

## 6. ROC Curve Comparison

In [ ]:
import matplotlib.pyplot as plt

y_prob_baseline = baseline_clf.predict_proba(X_test[TABULAR_FEATURES])[:, 1]
y_prob_graph    = graph_clf.predict_proba(X_test[ALL_FEATURES])[:, 1]

fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_baseline)
fpr_g, tpr_g, _ = roc_curve(y_test, y_prob_graph)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_b, tpr_b, label=f"Baseline (AUC={baseline_metrics['auc']:.3f})", linewidth=2)
ax.plot(fpr_g, tpr_g, label=f"Graph-Augmented (AUC={graph_metrics['auc']:.3f})", linewidth=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Baseline vs Graph-Augmented")
ax.legend(loc="lower right")
plt.tight_layout()
display(fig)

## 7. Confusion Matrix Deep-Dive

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_pred_baseline = baseline_clf.predict(X_test[TABULAR_FEATURES])
y_pred_graph    = graph_clf.predict(X_test[ALL_FEATURES])

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_baseline, ax=axes[0], cmap="Blues")
axes[0].set_title(f"Baseline (AUC={baseline_metrics['auc']:.3f})")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_graph, ax=axes[1], cmap="Oranges")
axes[1].set_title(f"Graph-Augmented (AUC={graph_metrics['auc']:.3f})")

plt.tight_layout()
display(fig)

---
## 8. The Money Slide

Translate model lift into estimated business impact.

In [ ]:
avg_fraud_loss     = 5000  # dollars per undetected fraud case
test_fraud_count   = y_test.sum()

baseline_caught    = (y_pred_baseline[y_test == 1] == 1).sum()
graph_caught       = (y_pred_graph[y_test == 1] == 1).sum()
additional_caught  = graph_caught - baseline_caught

print(f"Fraud cases in test set:        {test_fraud_count}")
print(f"Baseline caught:                {baseline_caught}  ({baseline_caught/test_fraud_count*100:.1f}%)")
print(f"Graph-augmented caught:         {graph_caught}  ({graph_caught/test_fraud_count*100:.1f}%)")
print(f"Additional fraud caught:        {additional_caught}")
print(f"Est. savings (@ ${avg_fraud_loss:,}/case): ${additional_caught * avg_fraud_loss:,.0f}")

---
## Key Takeaway

> Three graph algorithms from Neo4j GDS — computed in seconds in Aura —
> gave us more predictive lift than months of manual tabular feature engineering.

The graph features capture **structural signals** (centrality, clustering,
shared counterparties) that simply don't exist in flat transaction tables.

---

### Demo Flow Recap

| Step | Notebook | Where |
|------|----------|-------|
| 1. Generate synthetic fraud data | `00_setup_and_data` | Databricks |
| 2. Push to Neo4j Aura | `01_neo4j_ingest` | Databricks |
| 3. Run GDS algorithms | `02_aura_gds_guide` | Neo4j Aura Workspace |
| 4. Pull features & train models | **`03_pull_gold_tables`** | Databricks |

### The Bidirectional Loop
```
Delta Lake ──► Neo4j Aura ──► GDS (PageRank, Louvain, Similarity)
                                         │
Feature Store ◄── Training Table ◄── Spark Connector Read
     │
  ML Model  (baseline vs graph-augmented)
```